In [14]:
import pandas as pd
import os
from features_reindex import get_feature, read_data, read_data_timecut
import numpy as np
import pandas as pd
from sklearn import svm
from sklearn.model_selection import KFold, GridSearchCV
from rdkit.ML.Scoring.Scoring import CalcBEDROC
# from pseudo_label import select_pseudo_negatives
from sklearn.metrics import roc_auc_score
from sklearn.metrics import make_scorer
import os
import pickle
import gseapy as gp
# from concurrent.futures import ProcessPoolExecutor
# import functools
from multiprocessing import Pool
from collections import defaultdict
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.neighbors import NearestNeighbors
from scipy.linalg import logm, expm, eigh
from mygene import MyGeneInfo
from model_reindex_fusion_weights_uniport import eval_bagging

In [2]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm'

feature_list = ['uniport_ppi_2019','uniport_bio','uniport_seq','uniport_esm']
out_path = os.path.join(root,'results/temp')
time = 2019

In [73]:
merged_df = None
for feature in feature_list:
    feature_df = get_feature(root, feature)
    # Rename columns starting with 'feature'
    feature_df.rename(columns={
        col: f"{feature}_{col}" if col.startswith('feature') else col
        for col in feature_df.columns
    }, inplace=True)
    # Merge iteratively to avoid keeping all DataFrames
    if merged_df is None:
        merged_df = feature_df
    else:
        merged_df = pd.merge(merged_df, feature_df, on='string_id', how='inner')
    del feature_df  # Free memory
all_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/disgent_2020/timecut/dga_time_uniport.csv')
all_df = all_df[all_df['string_id'].isin(merged_df['string_id'])]

# all_df = all_df[all_df['score']>=0.7]

methods = ['random_negative']
disease = 'ICD10_D57'
print(disease,len(all_df[all_df['disease_id']==disease]))
df, y = read_data_timecut(disease, all_df, merged_df,time)

ICD10_D57 22


In [74]:
test_idx = df[df['test']==1].index
train_idx = df[y==1].index.difference(test_idx)
df.drop(columns='test', inplace=True)

train_pos_df = df.loc[train_idx]
test_pos_df = df.loc[test_idx]
neg_num = 5*len(train_pos_df)



In [75]:
gda = all_df[all_df['disease_id'] == disease]
gda[gda['string_id'].isin(test_pos_df.index.values)]

,disease_id,omim,hpo,do,disease_name,gene_id,entrz,esembl,uniport,score,first_pub_year,last_pub_year,ei,dsi,dpi,string_id
12672,ICD10_D57,"['OMIM_603903', 'OMIM_141900']",[],"['DO_0081445', 'DO_10923']","Anemia, Sickle Cell",CFB,629,['ENSG00000243649'],P00751,0.65,2022.0,NaN,0.857,0.459,0.913,P00751


In [76]:
gda[gda['string_id'].isin(train_pos_df.index.values)]['score'].mean(), gda[gda['string_id'].isin(test_pos_df.index.values)]['score'].mean()

(0.6833333333333333, 0.65)

In [70]:
test_pos_df.index.values

array(['O15360'], dtype=object)

In [71]:
feature_list

['uniport_ppi_2019', 'uniport_bio', 'uniport_seq', 'uniport_esm']

In [72]:
X_all = []
# for feature_name in feature_list:
# for width_expand in [0.005, 0.01, 0.1]:
for neg_ratio in [1,3,5,10]:
    neg_num = neg_ratio*len(train_pos_df)
    neg_df = df[y == 0]


    train_neg_df = neg_df.sample(n=neg_num, replace=True, random_state=42)
    train_df = pd.concat([train_pos_df, train_neg_df])
    train_index_loc = df.index.get_indexer(train_df.index)
    y_train = np.array([1] * len(train_pos_df) + [0] * len(train_neg_df))

    test_neg_df = neg_df.drop(train_neg_df.index)
    test_df = pd.concat([test_pos_df, test_neg_df])
    test_index_loc = df.index.get_indexer(test_df.index)
    y_test = np.array([1] * len(test_pos_df) + [0] * len(test_neg_df))

    for width_expand in [0.001,0.005, 0.01, 0.1,2,4,6]:
        feature_name = feature_list[2]
        select_columns = [col for col in df.columns if col.startswith(feature_name)]
        X_feature = df[select_columns].values
        X_all.append(X_feature)

        nbrs = NearestNeighbors(n_neighbors=2).fit(X_feature)
        distances, _ = nbrs.kneighbors(X_feature)
        avg_nn_dist = np.mean(distances[:, 1])  # skip self-distance
        gamma = 1 / (width_expand * avg_nn_dist ** 2)
        K_full = rbf_kernel(X_feature, X_feature, gamma=gamma)

        X_feature_train = K_full[np.ix_(train_index_loc, train_index_loc)]
        X_feature_test = K_full[np.ix_(test_index_loc,train_index_loc)]

        best_svm = svm.SVC(kernel='precomputed')
        best_svm.fit(X_feature_train, y_train)
        y_scores = best_svm.decision_function(X_feature_test)
        print(neg_ratio,width_expand, ': ', eval_bagging(y_scores,y_test)[1][-6])

1 0.001 :  0.4999038029885205
1 0.005 :  0.49996793432950687
1 0.01 :  0.4995831462835888
1 0.1 :  0.1488488424292952
1 2 :  0.007118578849483748
1 4 :  0.0030783043673443533
1 6 :  0.0041685371641121405
3 0.001 :  0.5000967679504549
3 0.005 :  0.5008063995871235
3 0.01 :  0.5013547513063673
3 0.1 :  0.11683117218243988
3 2 :  0.05780272240500617
3 4 :  0.07115669956776982
3 6 :  0.06889878072382427
5 0.001 :  0.5000973330737785
5 0.005 :  0.5009408863798586
5 0.01 :  0.5018168840438648
5 0.1 :  0.16014535072350922
5 2 :  0.10648238271364607
5 4 :  0.12588410875348777
5 6 :  0.1651417818441373
10 0.001 :  0.5001975373674854
10 0.005 :  0.502139988147758
10 0.01 :  0.5053335089221044
10 0.1 :  0.23757160729571347
10 2 :  0.17455718706788703
10 4 :  0.2927503786132877
10 6 :  0.3798643576743267


In [62]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

X_train = train_df.values
X_test = test_df.values


# Step 1: Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 2: Initialize and train Logistic Regression
log_reg = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

# Step 3: Predict probabilities for the test set
y_probs = log_reg.predict_proba(X_test_scaled)[:, 1]  # Get probability for class 1

# Step 4: Evaluate AUC-ROC
auc_score = roc_auc_score(y_test, y_probs)
print(f'AUC-ROC Score: {auc_score:.4f}')

# Optional: Print detailed classification report
y_pred = log_reg.predict(X_test_scaled)
print(classification_report(y_test, y_pred))


AUC-ROC Score: 0.4555
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     15484
           1       0.00      0.00      0.00         4

    accuracy                           0.97     15488
   macro avg       0.50      0.49      0.49     15488
weighted avg       1.00      0.97      0.99     15488

